# 01 — Exploratory Data Analysis (EDA)

**Nguồn dữ liệu**: MySQL → bảng `cleaned_cars` (đã load bởi ETL pipeline)

**Mục tiêu**:
- Kết nối DB, SELECT toàn bộ dữ liệu đã clean
- Phân tích phân phối giá, km, năm sản xuất
- Vẽ biểu đồ theo từng đặc trưng phân loại
- Tìm tương quan giữa các biến số

> ⚠️ **Yêu cầu**: Đã chạy `python -m src.etl_pipeline` để load data vào DB trước.

## 0. Import & Cấu hình

In [ ]:
import sys
from pathlib import Path

# Thêm thư mục gốc project vào sys.path
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Thư mục lưu hình ảnh output
OUTPUT_DIR = ROOT / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

from src.db_connector import DBConnector

warnings.filterwarnings('ignore')

# Style toàn cục
sns.set_theme(style='darkgrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.figsize': (10, 5),
    'font.family': 'DejaVu Sans',
})

print(f'✓ Import thành công')
print(f'✓ Output dir: {OUTPUT_DIR}')

## 1. Kết nối Database & Load dữ liệu

In [ ]:
# Kết nối MySQL và SELECT toàn bộ dữ liệu đã clean
db = DBConnector()
conn = db.get_connection()

df = pd.read_sql("""
    SELECT 
        id, ten_xe, gia, so_ghe, nam_sx,
        nhien_lieu, kieu_dang, tinh_trang,
        km_da_di, hop_so, xuat_xu, tinh_thanh,
        log_gia, log_km
    FROM cleaned_cars
    ORDER BY id
""", conn)

print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Tổng quan dữ liệu
print('=== Thông tin cột ===')
df.info()
print('\n=== Thống kê mô tả (số) ===')
df.describe()

In [ ]:
# Kiểm tra giá trị thiếu
null_counts = df.isnull().sum()
print('Giá trị NaN còn lại trong DB:')
print(null_counts[null_counts > 0] if null_counts.any() else '✓ Không có NaN')

## 2. Phân phối Giá xe

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Phân phối giá gốc
axes[0].hist(df['gia'] / 1e9, bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('Phân phối Giá (tỷ VND)', fontweight='bold')
axes[0].set_xlabel('Giá (tỷ VND)')
axes[0].set_ylabel('Số lượng')

# Phân phối log(giá)
axes[1].hist(df['log_gia'], bins=50, color='coral', edgecolor='white', alpha=0.85)
axes[1].set_title('Phân phối Log(Giá) — sau biến đổi', fontweight='bold')
axes[1].set_xlabel('log(Giá + 1)')
axes[1].set_ylabel('Số lượng')

plt.suptitle('Phân phối Giá xe trên oto.com.vn', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_01_price_distribution.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'✓ Đã lưu → output/eda_01_price_distribution.png')

## 3. Phân tích theo Tình trạng (Xe cũ / Xe mới)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Đếm số lượng
status_counts = df['tinh_trang'].value_counts()
axes[0].bar(status_counts.index, status_counts.values,
            color=['#2196F3', '#FF5722'], edgecolor='white', alpha=0.9)
axes[0].set_title('Số lượng Xe cũ vs Xe mới', fontweight='bold')
axes[0].set_ylabel('Số lượng')
for i, v in enumerate(status_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# Boxplot giá theo tình trạng
df_plot = df.copy()
df_plot['gia_ty'] = df_plot['gia'] / 1e9
df_plot.boxplot(column='gia_ty', by='tinh_trang', ax=axes[1],
                patch_artist=True,
                boxprops=dict(facecolor='lightblue'),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Phân phối Giá theo Tình trạng', fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('Giá (tỷ VND)')
plt.suptitle('')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_02_status_price.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'✓ Đã lưu → output/eda_02_status_price.png')

## 4. Xuất xứ & Kiểu dáng

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Xuất xứ
origin_counts = df['xuat_xu'].value_counts()
axes[0].barh(origin_counts.index, origin_counts.values,
             color=sns.color_palette('Set2', len(origin_counts)))
axes[0].set_title('Xuất xứ xe', fontweight='bold')
axes[0].set_xlabel('Số lượng')

# Kiểu dáng
style_counts = df['kieu_dang'].value_counts().head(8)
axes[1].barh(style_counts.index, style_counts.values,
             color=sns.color_palette('Set3', len(style_counts)))
axes[1].set_title('Kiểu dáng xe (Top 8)', fontweight='bold')
axes[1].set_xlabel('Số lượng')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_03_origin_style.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'✓ Đã lưu → output/eda_03_origin_style.png')

## 5. Ma trận tương quan

In [ ]:
num_cols = ['gia', 'so_ghe', 'nam_sx', 'km_da_di', 'log_gia', 'log_km']
corr = df[num_cols].corr()

plt.figure(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    linewidths=0.5, square=True,
    cbar_kws={'shrink': 0.8}
)
plt.title('Ma trận tương quan', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_04_correlation_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'✓ Đã lưu → output/eda_04_correlation_heatmap.png')

## 6. Giá vs Năm sản xuất & Km đã đi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

color_map = {'Xe cũ': '#2196F3', 'Xe mới': '#FF5722'}

for status, grp in df.groupby('tinh_trang'):
    axes[0].scatter(grp['nam_sx'], grp['gia'] / 1e9,
                    label=status, alpha=0.4, s=15,
                    color=color_map.get(status, 'gray'))

axes[0].set_title('Giá vs Năm sản xuất', fontweight='bold')
axes[0].set_xlabel('Năm SX')
axes[0].set_ylabel('Giá (tỷ VND)')
axes[0].legend()

for status, grp in df[df['tinh_trang'] == 'Xe cũ'].groupby('tinh_trang'):
    axes[1].scatter(grp['km_da_di'] / 1000, grp['gia'] / 1e9,
                    alpha=0.3, s=15, color='#2196F3', label='Xe cũ')

axes[1].set_title('Giá vs Km đã đi (Xe cũ)', fontweight='bold')
axes[1].set_xlabel('Km đã đi (nghìn km)')
axes[1].set_ylabel('Giá (tỷ VND)')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_05_price_vs_features.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'✓ Đã lưu → output/eda_05_price_vs_features.png')

## 7. Violin Plot: Giá theo Kiểu dáng

In [ ]:
# Chỉ lấy top kiểu dáng để biểu đồ rõ hơn
top_styles = df['kieu_dang'].value_counts().head(6).index
df_violin = df[df['kieu_dang'].isin(top_styles)].copy()
df_violin['gia_ty'] = df_violin['gia'] / 1e9

plt.figure(figsize=(12, 6))
sns.violinplot(
    data=df_violin, x='kieu_dang', y='gia_ty',
    palette='Set2', inner='box', cut=0
)
plt.title('Phân phối Giá theo Kiểu dáng xe', fontweight='bold', fontsize=13)
plt.xlabel('Kiểu dáng')
plt.ylabel('Giá (tỷ VND)')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_06_violin_style.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'✓ Đã lưu → output/eda_06_violin_style.png')

## 8. Đóng kết nối

In [ ]:
db.close()
print('✓ Đã đóng kết nối DB')
print(f'\nTất cả biểu đồ đã lưu tại: {OUTPUT_DIR}')